In [1]:
import sys
import logging
from pathlib import Path

import torch

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_fold
from utils.mlp_utils_lo import (
    set_seed,
    prepare_all_fold_tensors_lo,
    run_nested_random_search_lo,
    print_final_results_lo,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

23:08:46 [INFO] Device: cuda


In [2]:
CFG = {
    "task":        "lo",
    "dataset":     "kdr",
    "fp_type":     "ecfp4",
    "n_bits":      1024,
    "outer_folds": [1, 2, 3],
    "inner_k":     2,
    "random_state": GLOBAL_SEED,
}

In [ ]:
SEARCH_SPACE = {
    "n_layers":     [1, 2, 3],
    "n_nodes":      [64, 128, 256, 512, 1024],
    "r":            [0.5, 0.7, 1.0],
    "activation":   ["relu", "leaky_relu", "gelu", "silu"],  
    "dropout":      [0.0, 0.2, 0.3, 0.5],
    "batchnorm":    [True, False],
    "init":         ["kaiming", "xavier"],

    "lr":           [1e-4, 5e-4, 1e-3, 2e-3, 3e-3],
    "weight_decay": [0.0, 1e-5, 1e-4, 1e-3, 5e-3, 1e-2],
    "batch_size":   [64, 128, 256],
}

FIXED_HP = {
    "max_epochs": 100,
    "patience":   10,
    "grad_clip":  1.0,
}

N_ITER  = 150
N_SEEDS = 3

In [4]:
folds_data = {}

for fold_idx in CFG["outer_folds"]:
    train_df, test_df = load_fold(CFG["task"], CFG["dataset"], fold_idx)

    folds_data[fold_idx] = {"train": train_df, "test": test_df}

    logger.info(
        f"Fold {fold_idx} | "
        f"train={len(train_df)} "
        f"y_mean={train_df['value'].mean():.3f} "
        f"y_std={train_df['value'].std():.3f} | "
        f"test={len(test_df)} "
        f"n_clusters={test_df['cluster'].nunique()}"
    )

folds_tensors = prepare_all_fold_tensors_lo(
    cfg=CFG,
    folds_data=folds_data,
    logger=logger,
)

23:08:47 [INFO] Loaded lo/kdr fold 1: train=500, test=437
23:08:47 [INFO] Fold 1 | train=500 y_mean=6.769 y_std=0.894 | test=437 n_clusters=54
23:08:47 [INFO] Loaded lo/kdr fold 2: train=500, test=520
23:08:47 [INFO] Fold 2 | train=500 y_mean=6.774 y_std=0.856 | test=520 n_clusters=60
23:08:47 [INFO] Loaded lo/kdr fold 3: train=500, test=417
23:08:47 [INFO] Fold 3 | train=500 y_mean=6.739 y_std=0.865 | test=417 n_clusters=54
23:08:47 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kdr/ecfp4_train_1.npz
23:08:47 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kdr/ecfp4_test_1.npz
23:08:47 [INFO] Fold 1 | X_train: (500, 1024), X_test: (437, 1024) | y_mean=6.769 y_std=0.893
23:08:47 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kdr/ecfp4_train_2.npz
23:08:47 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kdr/ecfp4_test_2.npz
23:08:4

In [ ]:
# ── Block 4 ── Nested CV random search ─────────────────────────────────────

logger.info(f"Estimated time: ~{N_ITER * 6 * len(CFG['outer_folds']) / 60:.0f} min")

fold_results = run_nested_random_search_lo(
    cfg=CFG,
    folds_tensors=folds_tensors,
    folds_data=folds_data,
    search_space=SEARCH_SPACE,
    fixed_hp=FIXED_HP,
    n_iter=N_ITER,
    n_seeds=N_SEEDS,
    device=device,
    seed=GLOBAL_SEED,
    logger=logger,
)

print_final_results_lo(fold_results, title="MLP kdr Lo — ecfp4")

23:08:47 [INFO] Estimated time: ~45 min
23:08:47 [INFO] 
OUTER FOLD 1
23:08:48 [INFO]   [1/150] inner Spearman=0.5837 (2s) | L=2 N=512 r=0.7 dropout=0.3 lr=3e-03
23:08:49 [INFO]   [2/150] inner Spearman=0.5753 (1s) | L=3 N=128 r=1.0 dropout=0.2 lr=1e-03
23:08:49 [INFO]   [3/150] inner Spearman=0.4473 (0s) | L=1 N=1024 r=0.7 dropout=0.0 lr=5e-04
23:08:50 [INFO]   [4/150] inner Spearman=0.5625 (0s) | L=2 N=128 r=0.7 dropout=0.3 lr=3e-03
23:08:50 [INFO]   [5/150] inner Spearman=0.5169 (0s) | L=2 N=256 r=1.0 dropout=0.6 lr=3e-03
23:08:51 [INFO]   [6/150] inner Spearman=0.5340 (1s) | L=1 N=128 r=0.7 dropout=0.6 lr=1e-04
23:08:52 [INFO]   [7/150] inner Spearman=0.5572 (1s) | L=3 N=256 r=1.0 dropout=0.5 lr=1e-03
23:08:53 [INFO]   [8/150] inner Spearman=0.5515 (1s) | L=2 N=128 r=0.7 dropout=0.5 lr=5e-04
23:08:54 [INFO]   [9/150] inner Spearman=0.5191 (1s) | L=1 N=256 r=0.5 dropout=0.5 lr=1e-04
23:08:55 [INFO]   [10/150] inner Spearman=0.4920 (1s) | L=3 N=128 r=0.5 dropout=0.0 lr=5e-04
23:08:56


MLP DRD2 Lo — ecfp4
Fold 1: mean_spearman=0.1580
Fold 2: mean_spearman=0.2141
Fold 3: mean_spearman=0.1128

Aggregated metrics:
  mean_spearman_mean: 0.1616
  mean_spearman_std: 0.0414
  std_spearman_mean: 0.4612
  std_spearman_std: 0.0098
  mean_r2_mean: -0.931
  mean_r2_std: 0.3168
  mean_mae_mean: 1.0032
  mean_mae_std: 0.0058
  n_clusters_mean: 56.0
  n_clusters_std: 2.8284

Best hyperparameters per fold:
Fold 1: L=2 N=512 r=1.0 act=gelu dropout=0.0 bn=True init=kaiming lr=2e-03 wd=1e-03 bs=128
Fold 2: L=3 N=1024 r=0.5 act=relu dropout=0.0 bn=True init=kaiming lr=2e-03 wd=5e-03 bs=128
Fold 3: L=3 N=512 r=1.0 act=leaky_relu dropout=0.5 bn=True init=kaiming lr=2e-03 wd=1e-02 bs=256


{'mean_spearman_mean': np.float64(0.1616),
 'mean_spearman_std': np.float64(0.0414),
 'std_spearman_mean': np.float64(0.4612),
 'std_spearman_std': np.float64(0.0098),
 'mean_r2_mean': np.float64(-0.931),
 'mean_r2_std': np.float64(0.3168),
 'mean_mae_mean': np.float64(1.0032),
 'mean_mae_std': np.float64(0.0058),
 'n_clusters_mean': np.float64(56.0),
 'n_clusters_std': np.float64(2.8284)}